<h1>Modules Testing</h1>
<br>
<p>In this notebook will be build and test the modules, functions and flows using in the system.
At the moment of building the entire system, i thought the best way to actually define the flows and the modules is using the jupyter notebook.
</p>

In [2]:
from app.services.loader import load_data

# Each is actually a tuple (X_train, y_train), (X_test, y_test)
X, y = load_data('../data_test/train.csv', 'Survived')
print(f'Shape of X: {X[0].shape}')
print(f'Shape of y: {y[0].shape}')

Shape of X: (801, 11)
Shape of y: (90, 11)


In [31]:
from app.core.domain.feature_config import FeatureConfig

class EDA:
    ORDINAL_KEYWORDS: dict = {
        "low": 0,
        "medium": 1,
        "high": 2,
        "very low": -1,
        "very high": 3,
        "small": 0,
        "large": 2,
        "near": 0,
        "far": 2,
        "poor": 0,
        "fair": 1,
        "good": 2,
        "excellent": 3
    }

    def __init__(self, df: pd.DataFrame):
        self._inspect_data(df)

    def _inspect_data(self, df: pd.DataFrame) -> None:
        """
        Inspects the dataframe and creates a FeatureConfig instance
        for each column.

        :param df: Input dataframe.
        :return: None
        """
        self.num_rows, self.num_cols = df.shape
        self.feature_configs = []

        df = df.copy()  # Avoid modifying the original dataframe
        df = self._drop_redundant_columns(df)

        for col in df.columns:
            series = df[col]

            feature_type = self.detect_feature_type(series)

            cardinality = None
            encoding = None
            skewness = None
            zero_ratio = None
            suggested_transformation = None

            is_categorical = False
            is_numerical = False

            # -------- CATEGORICAL FEATURES --------
            if feature_type == "categorical":
                is_categorical = True
                cardinality = self.compute_cardinality(series)
                encoding = self.decide_categorical_encoding(series, cardinality)

            # -------- NUMERICAL FEATURES --------
            elif feature_type == "numerical":
                is_numerical = True
                dist_info = self.analyze_numerical_distribution(series)
                skewness = dist_info["skewness"]
                zero_ratio = dist_info["zero_ratio"]
                suggested_transformation = dist_info["suggested_transformation"]

            # -------- BINARY FEATURES --------
            elif feature_type == "binary":
                # Binary features are treated as numerical by default
                is_numerical = True

            feature_config = FeatureConfig(
                name=col,
                dtype=str(series.dtype),
                feature_type=feature_type,
                cardinality=cardinality,
                encoding=encoding,
                is_categorical=is_categorical,
                is_numerical=is_numerical,
                skewness=skewness,
                zero_ratio=zero_ratio,
                suggested_transformation=suggested_transformation
            )
            self.feature_configs.append(feature_config)


    @staticmethod
    def _drop_redundant_columns(
        df: pd.DataFrame,
        null_threshold: float = 0.9,
        unique_ratio_threshold: float = 0.95,
    ):
        columns_to_drop = []

        n_rows = len(df)

        for col in df.columns:
            series = df[col]

            # High null ratio
            if series.isnull().mean() > null_threshold:
                columns_to_drop.append(col)
                continue

            # Constant column
            if series.nunique(dropna=False) <= 1:
                columns_to_drop.append(col)
                continue

            # ID-like detection (it is supposed ONLY for object or integer types)
            if pd.api.types.is_object_dtype(series) or pd.api.types.is_integer_dtype(series):
                unique_ratio = series.nunique() / n_rows
                if unique_ratio > unique_ratio_threshold:
                    columns_to_drop.append(col)
                    continue

        return df.drop(columns_to_drop, axis=1)

    @staticmethod
    def detect_feature_type(
            series: pd.Series,
    ) -> str:
        """
        Detects the semantic type of feature.

        Possible outputs:
        - 'Binary'
        - 'Categorical'
        - 'Numerical'

        :param series: Pandas Series representing a feature column.
        :return: Detected feature type as a string.
        """
        s = series.dropna()

        # Boolean columns
        if pd.api.types.is_bool_dtype(s):
            return "binary"

        # Binary numeric or object (e.g., 0/1, yes/no)
        if s.nunique() == 2 and pd.api.types.is_object_dtype(s):
            return "binary"

        if pd.api.types.is_numeric_dtype(s) and s.nunique() == 2:
            return "binary"

        # Explicit categorical dtype
        if isinstance(s.dtype, pd.CategoricalDtype):
            return "categorical"

        # Object dtype (strings)
        if pd.api.types.is_object_dtype(s) and s.nunique() > 2:
            return "categorical"

        # Numeric columns
        if pd.api.types.is_numeric_dtype(s):
            return "numerical"

        # Fallback
        return "categorical"

    @staticmethod
    def compute_cardinality(series: pd.Series) -> int:
        """
        Computes the cardinality (number of unique values) of a feature.

        :param series: Pandas Series representing a feature column.
        :return: Number of unique non-null values.
        """
        return series.dropna().nunique()

    @staticmethod
    def suggest_categorical_encoding(
            cardinality: int,
            onehot_max_cardinality: int = 10
    ) -> str:
        """
        Suggests an encoding technique for categorical features.

        :param cardinality: Number of unique values in the feature.
        :param onehot_max_cardinality: Maximum cardinality to apply OneHotEncoding.
        :return: Suggested encoding method ('onehot' or 'ordinal').
        """
        if cardinality <= onehot_max_cardinality:
            return "onehot"
        return "ordinal"

    @staticmethod
    def detect_ordinal_semantics(series: pd.Series) -> bool:
        """
        Detects whether a categorical feature likely represents an ordinal variable
        based on semantic keywords.

        :param series: Pandas Series representing a categorical feature.
        :return: True if ordinal semantics are detected, False otherwise.
        """
        values = (
            series
            .dropna()
            .astype(str)
            .str.lower()
            .str.strip()
            .unique()
        )

        matches = 0
        for val in values:
            for keyword in EDA.ORDINAL_KEYWORDS:
                if keyword in val:
                    matches += 1
                    break

        # Heuristic: at least 2 ordinal-like values
        return matches >= 2

    @staticmethod
    def decide_categorical_encoding(
            series: pd.Series,
            cardinality: int,
            onehot_max_cardinality: int = 10
    ) -> str:
        """
        Decides the categorical encoding strategy.

        :param series: Pandas Series representing the feature.
        :param cardinality: Number of unique values.
        :param onehot_max_cardinality: Max cardinality for OneHot encoding.
        :return: Encoding strategy ('onehot' or 'ordinal').
        """
        if EDA.detect_ordinal_semantics(series):
            return "ordinal"

        if cardinality <= onehot_max_cardinality:
            return "onehot"

        return "ordinal"

    @staticmethod
    def analyze_numerical_distribution(series: pd.Series) -> dict:
        """
        Analyzes the distribution of a numerical feature to detect skewness
        and suggest transformations.

        :param series: Pandas Series representing a numerical feature.
        :return: Dictionary with distribution analysis.
        """
        s = series.dropna()

        skewness = s.skew()
        zero_ratio = (s == 0).mean()

        transformation = None

        if abs(skewness) > 1:
            if (s <= 0).any():
                transformation = "yeo-johnson"
            else:
                transformation = "log"

        return {
            "skewness": skewness,
            "zero_ratio": zero_ratio,
            "suggested_transformation": transformation
        }


In [5]:
import numpy as np
from sklearn.model_selection import cross_validate

from app.core.domain.experiments.experiment_result import ExperimentResult
from app.core.ml.pipeline_builder import PipelineBuilder
from app.core.ml.preprocessing_stage import PreprocessingBuilder


class Experiment:
    """
    Generic experiment runner.

    This class is intentionally kept flexible so it can be reused for:
    - Feature selection
    - Model selection
    - Hyperparameter tuning
    - Feature engineering experiments

    It does NOT decide which model to use.
    It does NOT decide which experiment is better.
    It only executes and returns results.
    """

    def __init__(
            self,
            name: str,
            pipeline_builder: PipelineBuilder,
            scoring: Union[str, List[str]],
            cv: int = 5,
            metadata: Dict[str, Any] | None = None
    ):
        """
        :param name: Unique experiment name.
        :param pipeline_builder: Responsible for constructing the sklearn pipeline.
        :param scoring: Scoring metrics for cross-validation.
        :param cv: Number of cross-validation folds.
        :param metadata: Optional experiment configuration for tracking.
        """
        self.name = name
        self.pipeline_builder = pipeline_builder
        self.scoring = scoring
        self.cv = cv
        self.metadata = metadata or {}

    def run(self,
            X: pd.DataFrame,
            y: pd.Series) -> ExperimentResult:
        """
        Executes cross-validation and fits the final model.

        :param X: Feature dataframe.
        :param y: Target series.
        :return: ExperimentResult object.
        """
        pipeline = self.pipeline_builder.build()
        scores = cross_validate(
            pipeline,
            X,
            y,
            scoring=self.scoring,
            cv=self.cv,
            n_jobs=-1,
            return_train_score=True
        )

        # Aggregate metrics properly
        mean_metrics = {}

        for metric_name, values in scores.items():

            if metric_name.startswith("train_") or metric_name.startswith("test_"):

                mean_value = float(np.mean(values))

                # Fix sklearn negative regression metrics
                if metric_name.endswith(("neg_mean_squared_error",
                                         "neg_mean_absolute_error")):
                    mean_value = -mean_value

                mean_metrics[metric_name] = mean_value

        # Fit the final pipeline on full dataset (artifact ready)
        pipeline.fit(X, y)

        return ExperimentResult(
            name=self.name,
            pipeline=pipeline,
            metrics=mean_metrics,
            config={
                "cv": self.cv,
                "scoring": self.scoring,
                **self.metadata
            },
            coef=pipeline.named_steps["model"].coef_
        )


In [6]:
from sklearn.pipeline import Pipeline
from dataclasses import dataclass


@dataclass
class ExperimentResult:
    name: str
    pipeline: Pipeline
    metrics: Dict[str, float]
    config: Dict[str, Any]
    coef: Dict[str, Any] | None = None

In [24]:
from sklearn.base import BaseEstimator
from typing import Tuple


class PipelineBuilder:
    def __init__(
            self,
            steps: List[Tuple[str, BaseEstimator]]
    ):
        self.steps = self._validate_steps(steps)

    def build(self) -> Pipeline:
        steps = self.steps
        return Pipeline(steps)

    def add_step(self, step: Tuple[str, BaseEstimator]):
        self.steps.append(step)

    @staticmethod
    def _validate_steps(steps: List[Tuple[str, BaseEstimator]]):
        if not steps:
            raise ValueError("Pipeline must have at least one step.")
        for name, estimator in steps:
            if not isinstance(name, str):
                raise TypeError(f"Step name must be a string. Got {type(name)}")
            if not isinstance(estimator, BaseEstimator):
                raise TypeError(f"Step estimator must be a sklearn BaseEstimator. Got {type(estimator)}")
        return steps

In [8]:
from sklearn.linear_model import ElasticNet

pipeline = PipelineBuilder(
    steps=[("ElasticNet", ElasticNet())]
).build()
pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('ElasticNet', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"alpha alpha: float, default=1.0Constant that multiplies the penalty terms. Defaults to 1.0.See the notes for the exact mathematical meaning of thisparameter. ``alpha = 0`` is equivalent to an ordinary least square,solved by the :class:`LinearRegression` object. For numericalreasons, using ``alpha = 0`` with the ``Lasso`` object is not advised.Given this, you should use the :class:`LinearRegression` object.",1.0
,"l1_ratio l1_ratio: float, default=0.5The ElasticNet mixing parameter, with ``0 <= l1_ratio <= 1``. For``l1_ratio = 0`` the penalty is an L2 penalty. ``For l1_ratio = 1`` itis an L1 penalty. For ``0 < l1_ratio < 1``, the penalty is acombination of L1 and L2.",0.5
,"fit_intercept fit_intercept: bool, default=TrueWhether the intercept should be estimated or not. If ``False``, thedata is assumed to be already centered.",True
,"precompute precompute: bool or array-like of shape (n_features, n_features), default=FalseWhether to use a precomputed Gram matrix to speed upcalculations. The Gram matrix can also be passed as argument.For sparse input this option is always ``False`` to preserve sparsity.Check :ref:`an example on how to use a precomputed Gram Matrix in ElasticNet`for details.",False
,"max_iter max_iter: int, default=1000The maximum number of iterations.",1000
,"copy_X copy_X: bool, default=TrueIf ``True``, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-4The tolerance for the optimization: if the updates are smaller or equal to``tol``, the optimization code checks the dual gap for optimality and continuesuntil it is smaller or equal to ``tol``, see Notes below.",0.0001


<p>experiment_definition.py</p>

In [25]:
from dataclasses import dataclass
from typing import Callable, Dict


@dataclass
class ExperimentDefinition:
    """
    Declarative experiment configuration.

    This object describes how to build an experiment
    but does not execute it.
    """

    name: str
    stage: str
    builder: Callable[..., PipelineBuilder]
    metadata: Dict[str, Any] | None = None


<p>experiments/feature_selection.py</p>

In [32]:
from sklearn.linear_model import ElasticNet
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import ExtraTreesClassifier

def elasticnet_selector_builder(preprocessing):

    selector = SelectFromModel(
        ElasticNet(alpha=0.1)
    )

    model = LogisticRegression(max_iter=1000)

    return PipelineBuilder(
        steps=[
            ("preprocessing", preprocessing),
            ("feature_selection", selector),
            ("model", model)
        ]
    )
def l1_logistic_selector_builder(preprocessing):

    selector = SelectFromModel(
        LogisticRegression(
            C=1.0,
            solver="liblinear",
            max_iter=1000
        )
    )

    model = LogisticRegression(max_iter=1000)

    return PipelineBuilder(
        steps=[
            ("preprocessing", preprocessing),
            ("feature_selection", selector),
            ("model", model)
        ]
    )

def random_forest_selector_builder(preprocessing):

    selector = SelectFromModel(
        RandomForestClassifier(
            n_estimators=200,
            random_state=42
        ),
        threshold="median"
    )

    model = LogisticRegression(max_iter=1000)

    return PipelineBuilder(
        steps=[
            ("preprocessing", preprocessing),
            ("feature_selection", selector),
            ("model", model)
        ]
    )

def extratrees_selector_builder(preprocessing):

    selector = SelectFromModel(
        ExtraTreesClassifier(
            n_estimators=200,
            random_state=42
        ),
        threshold="median"
    )

    model = LogisticRegression(max_iter=1000)

    return PipelineBuilder(
        steps=[
            ("preprocessing", preprocessing),
            ("feature_selection", selector),
            ("model", model)
        ]
    )

def no_selector_builder(preprocessing):

    model = LogisticRegression(max_iter=1000)

    return PipelineBuilder(
        steps=[
            ("preprocessing", preprocessing),
            ("model", model)
        ]
    )

FEATURE_SELECTION_EXPERIMENTS = [

    ExperimentDefinition(
        name="no_selector",
        stage="feature_selection",
        builder=no_selector_builder,
        metadata={"selector": None}
    ),

    ExperimentDefinition(
        name="elasticnet_selector",
        stage="feature_selection",
        builder=elasticnet_selector_builder,
        metadata={"selector": "ElasticNet", "alpha": 0.1}
    ),

    ExperimentDefinition(
        name="l1_logistic_selector",
        stage="feature_selection",
        builder=l1_logistic_selector_builder,
        metadata={"selector": "LogisticL1"}
    ),

    ExperimentDefinition(
        name="rf_selector",
        stage="feature_selection",
        builder=random_forest_selector_builder,
        metadata={"selector": "RandomForest"}
    ),

    ExperimentDefinition(
        name="extratrees_selector",
        stage="feature_selection",
        builder=extratrees_selector_builder,
        metadata={"selector": "ExtraTrees"}
    ),
]


<p>experiments/model_selection.py</p>

In [27]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier


def logistic_builder(preprocessing):

    return PipelineBuilder(
        steps=[
            ("preprocessing", preprocessing),
            ("model", LogisticRegression(max_iter=1000))
        ]
    )


def random_forest_builder(preprocessing):

    return PipelineBuilder(
        steps=[
            ("preprocessing", preprocessing),
            ("model", RandomForestClassifier(n_estimators=200))
        ]
    )


MODEL_SELECTION_EXPERIMENTS = [

    ExperimentDefinition(
        name="logistic_regression",
        stage="model_selection",
        builder=logistic_builder,
        metadata={"model": "LogisticRegression"}
    ),

    ExperimentDefinition(
        name="random_forest",
        stage="model_selection",
        builder=random_forest_builder,
        metadata={"model": "RandomForestClassifier"}
    ),

]


<p>registry.py</p>

In [28]:
from typing import List


_STAGE_REGISTRY = {
    "feature_selection": FEATURE_SELECTION_EXPERIMENTS,
    "model_selection": MODEL_SELECTION_EXPERIMENTS,
}

def get_stage_experiments(stage: str) -> List[ExperimentDefinition]:
    """
    Returns all experiment definitions registered for a given stage.
    """

    if stage not in _STAGE_REGISTRY:
        raise ValueError(f"Stage '{stage}' not registered.")

    return _STAGE_REGISTRY[stage]


<p>main.py</p>

In [47]:
from typing import Union

@dataclass(frozen=True)
class ProjectConfig:
    problem_type: str
    scoring: list[str]
    random_state: int
    priority_metrics: Union[str, list[str]] = None


In [48]:
from dataclasses import dataclass, field
from typing import Any

@dataclass
class RunContext:
    """
    Stores the state of a full experimentation workflow.
    """
    config: ProjectConfig

    # TODO: Create a ENUM to avoid typos when saving stages
    current_stage: str | None = None

    # This have to save the results of each stage, to be used in the next stages and for final comparison.
    # saving the stage name in the best metrics, also the path where the final artifact was saved
    stage_results: dict[str, str] = field(default_factory=dict)

    metadata: dict[str, Any] = field(default_factory=dict)

    stage_metrics: dict[str, dict] = field(default_factory=dict)



In [52]:
class Evaluator:
    """
    Tracks, compares and formats experiment results.
    """

    def __init__(self,
                 priority_metrics: Union[str, list[str]],
                 ):
        self.metric = priority_metrics
        self.results: list[ExperimentResult] = []

    def add_result(self, result: ExperimentResult) -> None:
        self.results.append(result)

    def get_best(self) -> ExperimentResult:
        if not self.results:
            raise RuntimeError("No experiments have been evaluated.")

        return max(
            self.results,
            key=lambda r: r.metrics[self.metric]
        )

    def summary(self) -> list[dict]:
        """
        Returns a summary of all experiments.
        """
        return [
            {
                "model": r.config["model"],
                self.metric: r.metrics[self.metric]
            }
            for r in self.results
        ]

In [54]:

from app.core.context.problems_type import ProblemsType

import pandas as pd

from app.core.eda_handler import EDA


class Orchestrator:
    def __init__(self,
                 X: tuple[pd.DataFrame, pd.Series],
                 y: tuple[pd.DataFrame, pd.Series],
                 context: RunContext
                 ) -> None:
        self.X = X
        self.y = y

        self.context = context

        self.metrics = self.context.config.priority_metrics
        self.problems_type = self.context.config.problem_type

    def flow(self):
        eda = EDA(self.X[0])
        self.feature_configs = eda.feature_configs

        self.feature_selection_experiment()

    def feature_selection_experiment(self):
        #evaluator = Evaluator()

        stage_name = "feature_selection"

        definitions = get_stage_experiments(stage_name)
        preprocessing = PreprocessingBuilder(
                    feature_configs=self.feature_configs
                ).build()
        results = []

        for definition in definitions:

            builder = definition.builder(preprocessing)
            experiment = Experiment(
                name=definition.name,
                pipeline_builder=builder,
                scoring=["f1", "accuracy"],
                cv=5,
                metadata=definition.metadata
            )

            result = experiment.run(self.X[0],
                                    self.X[1])
            print(f'Result: {result}')
            results.append(result)


In [55]:
def init_context():
    return (RunContext
        (
        ProjectConfig
        (ProblemsType.CLASSIFICATION,
         ["f1", "accuracy"],
         42))
    )

context = init_context()
Orchestrator(X,y, context).flow()


C:\Users\ville\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Result: ExperimentResult(name='no_selector', pipeline=Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['PassengerId', 'Pclass',
                                                   'Age', 'SibSp', 'Parch',
                                                   'Fare']),
                                                 ('cat_onehot',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                              

C:\Users\ville\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\ville\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
C:\Users\ville\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also

Result: ExperimentResult(name='l1_logistic_selector', pipeline=Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['PassengerId', 'Pclass',
                                                   'Age', 'SibSp', 'Parch',
                                                   'Fare']),
                                                 ('cat_onehot',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                     